In [1]:
# Cell 1 — Imports & setup
import sys
from pathlib import Path

# Menambahkan direktori root proyek ke sys.path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Mengimpor modul baru yang sudah diselaraskan dengan config.py
from src.transformers import load_june_logs, clean_logs, apply_feature_engineering, deduplicate_time_window
from config import (
    PARQUET_CLEANED,
    PARQUET_DEDUPED,
    CSV_DEDUPED,
    CSV_DEDUPED_SAMPLE,
    DATA_PROC_DIR,
    FIGURES_DIR,
)

DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Cell 2 — Load Raw Data & Eksekusi Pembersihan (Flattening)
df_raw = load_june_logs()
print(f"Total baris data mentah (raw) : {len(df_raw):,}")

# Eksekusi fungsi flattening dan drop null pada kolom kritis
df_clean = clean_logs(df_raw)
print(f"Total baris setelah cleaning  : {len(df_clean):,}")
print(f"Fitur hasil flattening        : {list(df_clean.columns)}")

# Simpan data flat sebagai checkpoint fase penyiapan awal
df_clean.to_parquet(PARQUET_CLEANED, index=False)
print(f"✓ Berkas sukses disimpan ke   : {PARQUET_CLEANED}")

Total baris data mentah (raw) : 429,269
Total baris setelah cleaning  : 422,610
Fitur hasil flattening        : ['timestamp', 'agent_name', 'rule_level', 'rule_description', 'rule_id', 'rule_groups', 'decoder_name', 'location']
✓ Berkas sukses disimpan ke   : e:\Aditya\Lecture\TUGAS-AKHIR\Clustering_Wazuh\data\processed\cleaned.parquet


In [3]:
# Cell 3 — Rekayasa Fitur Tahap 1 (Temporal Feature Engineering)
df_fe = apply_feature_engineering(df_clean)

print("Sampel hasil rekayasa fitur temporal:")
print(df_fe[["timestamp", "date", "hour", "hour_group"]].head())
print("\nDistribusi fitur hour_group:")
print(df_fe["hour_group"].value_counts())

Sampel hasil rekayasa fitur temporal:
                         timestamp        date  hour hour_group
0 2025-06-15 00:00:02.419000+00:00  2025-06-15     0  dini_hari
1 2025-06-15 00:00:07.365000+00:00  2025-06-15     0  dini_hari
2 2025-06-15 00:00:14.427000+00:00  2025-06-15     0  dini_hari
3 2025-06-15 00:00:14.427000+00:00  2025-06-15     0  dini_hari
4 2025-06-15 00:00:35.152000+00:00  2025-06-15     0  dini_hari

Distribusi fitur hour_group:
hour_group
dini_hari    155593
pagi          92992
malam         88347
siang         85678
Name: count, dtype: int64


In [4]:
# Cell 4 — Time-Window Aggregation (Deduplikasi Temporal)
deduped = deduplicate_time_window(df_fe)

print("=" * 60)
print(f"Volume log sebelum aglomerasi : {len(df_fe):,} baris")
print(f"Volume log setelah aglomerasi : {len(deduped):,} baris")
print(f"Persentase kompresi spam log  : {(1 - len(deduped)/len(df_fe))*100:.2f}%")
print("=" * 60)
print("\n10 Baris teratas hasil kompresi jendela waktu:")
print(deduped.head(10).to_string(index=False))

Volume log sebelum aglomerasi : 422,610 baris
Volume log setelah aglomerasi : 14,174 baris
Persentase kompresi spam log  : 96.65%

10 Baris teratas hasil kompresi jendela waktu:
 agent_name rule_id decoder_name  rule_level               location  alert_count hour_group
e-kuesioner   19004          sca           7                    sca            1       pagi
e-kuesioner   19004          sca           7                    sca            1  dini_hari
e-kuesioner   19004          sca           7                    sca            1      siang
e-kuesioner   19007          sca           7                    sca          104       pagi
e-kuesioner   19008          sca           3                    sca           88       pagi
e-kuesioner   19009          sca           3                    sca           15       pagi
e-kuesioner   19010          sca           3                    sca            1  dini_hari
e-kuesioner   19011          sca           9                    sca            1      

In [5]:
print(deduped.columns.tolist())
print(f"Shape: {deduped.shape}")

['agent_name', 'rule_id', 'decoder_name', 'rule_level', 'location', 'alert_count', 'hour_group']
Shape: (14174, 7)


In [8]:
# Cell 6 — Validasi Distribusi 7 Fitur K-Modes Clustering
cluster_features = ["agent_name", "rule_id", "decoder_name", "rule_level", "location", "hour_group"]

print("=== VERIFIKASI KESIAPAN FITUR K-MODES ===")
for col in cluster_features:
    print(f"\nFitur: {col} (Total Unik: {deduped[col].nunique()})")
    print(deduped[col].value_counts().head(5))
    print("-" * 40)

=== VERIFIKASI KESIAPAN FITUR K-MODES ===

Fitur: agent_name (Total Unik: 4)
agent_name
sads           6033
pusatkarir     5567
siput          1322
e-kuesioner    1252
Name: count, dtype: int64
----------------------------------------

Fitur: rule_id (Total Unik: 49)
rule_id
500554    4015
31101     3919
31151     1855
52507     1307
31122      958
Name: count, dtype: int64
----------------------------------------

Fitur: decoder_name (Total Unik: 15)
decoder_name
web-accesslog         6952
syscheck_new_entry    4091
freshclam             1307
nginx-errorlog         551
syscheck_deleted       370
Name: count, dtype: int64
----------------------------------------

Fitur: rule_level (Total Unik: 9)
rule_level
5     4891
12    4015
3     2189
10    1867
7      957
Name: count, dtype: int64
----------------------------------------

Fitur: location (Total Unik: 12)
location
/var/log/nginx/access.log    6952
syscheck                     4726
journald                     1510
/var/log/nginx/e

In [9]:
# Cell 7 — Simpan Hasil Final Fase Data Preparation
deduped.to_parquet(PARQUET_DEDUPED, index=False)
deduped.to_csv(CSV_DEDUPED, index=False)
deduped.head(20).to_csv(CSV_DEDUPED_SAMPLE, index=False)

print("=== FASILITAS ARSIP DATA PREPARATION SUKSES ===")
print(f"Berkas Parquet Utama  : {PARQUET_DEDUPED}")
print(f"Berkas CSV Pendukung  : {CSV_DEDUPED}")
print(f"Berkas Sampel Draf    : {CSV_DEDUPED_SAMPLE}")
print(f"Dimensi Matriks Final : {deduped.shape}")

=== FASILITAS ARSIP DATA PREPARATION SUKSES ===
Berkas Parquet Utama  : e:\Aditya\Lecture\TUGAS-AKHIR\Clustering_Wazuh\data\processed\deduped.parquet
Berkas CSV Pendukung  : e:\Aditya\Lecture\TUGAS-AKHIR\Clustering_Wazuh\data\processed\deduped.csv
Berkas Sampel Draf    : e:\Aditya\Lecture\TUGAS-AKHIR\Clustering_Wazuh\data\processed\deduped_sample20.csv
Dimensi Matriks Final : (14174, 7)
